# Data Cleaning & Visualization Project
This Jupyter Notebook demonstrates a complete, interactive, end-to-end data cleaning, preprocessing, and exploratory data analysis (EDA) pipeline.

### Objectives:
1. **Load and Profile** a raw, messy dataset.
2. **Standardize Column Headers** and text columns.
3. **Detect and Resolve Duplicates**.
4. **Parse and Convert Data Types** (dates, currency strings, custom string markers like "Free").
5. **Diagnose and Impute Missing Values** using customizable strategies (mean, median, mode, constant, ffill).
6. **Analyze and Prune Outliers** using the Interquartile Range (IQR) technique.
7. **Perform EDA** and render professional visualization plots.
8. **Export the Clean Dataset** and save a summary audit report.

## 1. Setup & Import Modules
We add the `src/` directory to the system path to load our custom pipeline classes.

In [ ]:
import os
import sys
# Set environment variables to limit OpenBLAS threads to 1 and prevent deadlock on import
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add source directory to import project modules
sys.path.append(os.path.abspath("../src"))

from generate_raw_data import generate_messy_dataset
from load_data import DataLoader
from cleaning import DataCleaner
from visualization import DataVisualizer
from report import ReportGenerator

%matplotlib inline
sns.set_theme(style="whitegrid")

## 2. Generate and Load Raw Messy Dataset
If the raw dataset does not exist, we programmatically trigger our messy data generator.

In [ ]:
raw_csv_path = "../data/raw_dataset.csv"

# Generate messy data if not present
if not os.path.exists(raw_csv_path):
    generate_messy_dataset(raw_csv_path, num_rows=1000)
else:
    print(f"Raw dataset already exists at {raw_csv_path}")

In [ ]:
# Initialize DataLoader
loader = DataLoader(raw_csv_path)
df_raw = loader.load_data()
loader.print_summary()

## 3. Data Cleaning Pipeline
We initialize our modular cleaning engine and step through the data preparation actions.

In [ ]:
# Initialize DataCleaner
cleaner = DataCleaner(df_raw)

# Step 3.1: Clean headers
cleaner.clean_column_names()
print("Standardized Column Headers:")
print(cleaner.df.columns.tolist())

In [ ]:
# Step 3.2: Convert and clean data types
# Handle price currency strings, commas, and keywords like 'Free'
cleaner.convert_numeric_column("sale_price", replace_rules={"Free": "0.0", "unknown": np.nan})
cleaner.convert_numeric_column("quantity")
cleaner.convert_numeric_column("customer_rating")

# Parse order dates to standard ISO datetime
cleaner.convert_date_column("order_date")

print("Columns after parsing and conversions:")
print(cleaner.df.dtypes)

In [ ]:
# Step 3.3: Standardize inconsistent categories and formatting
category_replacements = {
    "electrnics": "Electronics", "electronics": "Electronics", "ELECTronics": "Electronics",
    "home & kitchen": "Home & Kitchen", "HOME & KITCHEN": "Home & Kitchen", "Home": "Home & Kitchen",
    "clothing": "Clothing", "CLOTHing": "Clothing", "Clothes": "Clothing",
    "books": "Books", "Bks": "Books"
}
location_replacements = {
    "new york": "New York", "NEW YORK": "New York",
    "chicago": "Chicago", "CHICAGO": "Chicago",
    "los angeles": "Los Angeles", "LOS ANGELES": "Los Angeles",
    "houston": "Houston", "HOUSTON": "Houston"
}

cleaner.standardize_text_column("customer_name")
cleaner.standardize_text_column("product_category", replacement_map=category_replacements)
cleaner.standardize_text_column("store_location", replacement_map=location_replacements)

print("Unique Categories:", cleaner.df["product_category"].dropna().unique())
print("Unique Locations:", cleaner.df["store_location"].dropna().unique())

In [ ]:
# Step 3.4: Remove Duplicate Rows
duplicates_removed = cleaner.remove_duplicates()
print(f"Removed duplicate records: {duplicates_removed}")

## 4. Missing Values Resolution
We inspect the missing value distribution and apply customized imputation strategies.

In [ ]:
# Check missingness report
missing_before = cleaner.detect_missing_values()
print(missing_before)

In [ ]:
# Apply customized imputation
cleaner.impute_missing_values("transaction_id", strategy="drop")
cleaner.impute_missing_values("customer_id", strategy="constant", fill_value="C-Unknown")
cleaner.impute_missing_values("customer_name", strategy="constant", fill_value="Unknown Customer")
cleaner.impute_missing_values("product_category", strategy="mode")
cleaner.impute_missing_values("store_location", strategy="mode")
cleaner.impute_missing_values("sale_price", strategy="median")
cleaner.impute_missing_values("quantity", strategy="median")
cleaner.impute_missing_values("customer_rating", strategy="mean")
cleaner.impute_missing_values("order_date", strategy="ffill")

# Ensure no missing values remain
print("Remaining missing values:")
print(cleaner.df.isnull().sum())

## 5. Outlier Detection & Logical Validation
We detect and handle outliers via the Interquartile Range (IQR) technique and enforce domain integrity rules (positive prices, rating bounds).

In [ ]:
# Treat outliers using IQR
rating_outliers = cleaner.remove_outliers_iqr("customer_rating")
price_outliers = cleaner.remove_outliers_iqr("sale_price")
qty_outliers = cleaner.remove_outliers_iqr("quantity")

print(f"IQR Outliers Removed: Ratings ({rating_outliers}), Prices ({price_outliers}), Quantities ({qty_outliers})")

In [ ]:
# Force enforce domain logical bounds
cleaner.df = cleaner.df[cleaner.df["quantity"] > 0].reset_index(drop=True)
cleaner.df = cleaner.df[cleaner.df["sale_price"] >= 0].reset_index(drop=True)
cleaner.df = cleaner.df[(cleaner.df["customer_rating"] >= 1.0) & (cleaner.df["customer_rating"] <= 5.0)].reset_index(drop=True)

# Cast quantity to integer
cleaner.df["quantity"] = cleaner.df["quantity"].astype(int)

df_clean = cleaner.get_cleaned_data()
print(f"Cleaning complete. Final dimensions: {df_clean.shape}")

## 6. Exploratory Data Analysis & Visualizations
We generate and inspect selected visualizations directly inside the notebook environment.

In [ ]:
# 1. Sale Price Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_clean["sale_price"], kde=True, color="#4A90E2")
plt.title("Distribution of Cleaned Sale Prices", fontsize=15, fontweight="bold")
plt.xlabel("Sale Price ($)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# 2. Product Category Share
plt.figure(figsize=(8, 8))
cat_counts = df_clean["product_category"].value_counts()
plt.pie(cat_counts, labels=cat_counts.index, autopct="%1.1f%%", startangle=140, 
        colors=["#2B2D42", "#4A90E2", "#2ECC71", "#FF6B6B"])
plt.title("Product Category Distribution", fontsize=15, fontweight="bold")
plt.show()

In [ ]:
# 3. Revenue Trend over Time
plt.figure(figsize=(12, 5))
daily_sales = df_clean.groupby("order_date")["sale_price"].sum().reset_index()
daily_sales = daily_sales.sort_values(by="order_date")
daily_sales["smoothed"] = daily_sales["sale_price"].rolling(window=7, min_periods=1).mean()

sns.lineplot(x="order_date", y="smoothed", data=daily_sales, color="#2ECC71", linewidth=2.5)
plt.title("Revenue Trend over Time (7-Day Moving Avg)", fontsize=15, fontweight="bold")
plt.xlabel("Order Date")
plt.ylabel("Revenue ($)")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# 4. Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_clean[["sale_price", "quantity", "customer_rating"]].corr(), 
            annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Matrix", fontsize=15, fontweight="bold")
plt.show()

## 7. Save Cleaned Outputs & Compile Reports

In [ ]:
# Create output directories
os.makedirs("../output", exist_ok=True)

# Export dataset
clean_dataset_path = "../output/cleaned_dataset.csv"
df_clean.to_csv(clean_dataset_path, index=False)
print(f"Cleaned dataset saved to: {clean_dataset_path}")

# Export summary report
reporter = ReportGenerator(df_clean, cleaner.logs)
report_txt_path = "../output/summary_report.txt"
reporter.save_report(report_txt_path)

# Print a subset of report to confirm contents
print("\n--- REPORT PREVIEW ---")
print("\n".join(reporter.generate_report_text().split("\n")[:35]))